# LLM BD — Train Your Own Bangla LLM From Scratch in 2 Hours

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnabri-ops/llm-bd/blob/main/train.ipynb)

A 64M Bangla language model, trained from scratch in pure PyTorch. No API keys, no fine-tuning of someone else's model — you build the whole thing.

**Run this notebook top to bottom on a free Colab GPU (T4).**

In [ ]:
# 1. Clone + install
!git clone https://github.com/tunnabri-ops/llm-bd.git
%cd llm-bd
!pip install -q -r requirements.txt

In [ ]:
# 2. Build the Bangla datasets (207 pretrain sentences + 132 SFT pairs)
!python dataset/build_dataset.py

In [ ]:
# 3. Train the Bangla BPE tokenizer (vocab 8192)
!python train_tokenizer.py

In [ ]:
# 4. Tiny CPU smoke test first (works even without GPU)
!python trainer/train_pretrain.py --tiny --steps 60 --out out/tiny_demo.pth

In [ ]:
# 5. Full run: llm-bd-64M pretraining (GPU — about 2 hours on a T4 / 3090)
# !python trainer/train_pretrain.py --config 64M --steps 2000 --device cuda --out checkpoints/llm-bd-64M/pretrain.pth

In [ ]:
# 6. SFT — teach it to answer in Bangla
# !python trainer/train_sft.py --init checkpoints/llm-bd-64M/pretrain.pth --steps 300 --device cuda --out out/sft_64m.pth

In [ ]:
# 7. Chat with your model
from model.model import LLM
from model.tokenizer import BanglaTokenizer
import torch
tok = BanglaTokenizer('model/tokenizer.json')
model = LLM.load('out/tiny_demo.pth')
for prompt in ['ঢাকা বাংলাদেশের', 'বাংলা ভাষা']:
    ids = torch.tensor([tok.encode(prompt)])
    out = model.generate(ids, max_new_tokens=40, temperature=0.8)
    print(prompt, '->', tok.decode(out[0].tolist()))

In [ ]:
# 8. Perplexity on held-out Bangla text
!python eval_llm.py --checkpoint out/tiny_demo.pth --perplexity